# Response Streaming

A full response can take 10–30 seconds to generate. Without streaming, the user stares at a spinner until the whole thing arrives. **Streaming** sends the response back in small chunks as it's generated, so text appears word-by-word — a much more responsive feel.

All the chunks are part of **one** request to Claude. When streaming, Claude emits a series of typed events:

| Event | Meaning |
|-------|---------|
| `message_start` | A new message is beginning |
| `content_block_start` | Start of a content block (text, tool use, etc.) |
| `content_block_delta` | **A chunk of generated text** — the part you display |
| `content_block_stop` | The current content block is done |
| `message_delta` | Message-level update (stop reason, usage) |
| `message_stop` | End of the message |

> Back on `claude-sonnet-5` for this lesson — streaming has no deprecated parameters.

## Setup

In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"


def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


print(f"Ready. Model: {model}")

Ready. Model: claude-sonnet-5


## 1. Raw events — `stream=True`

The lowest-level view: pass `stream=True` to `messages.create()` and iterate. This prints every event object so you can see the shapes described in the table above.

In [2]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True,
)

for event in stream:
    print(event.type)   # print just .type; drop `.type` to see the full event objects

message_start
content_block_start
content_block_delta
content_block_delta
content_block_delta
content_block_delta
content_block_stop
message_delta
message_stop


## 2. Simplified text streaming — `stream.text_stream`

Usually you don't want to parse events by hand. The `client.messages.stream(...)` context manager exposes `text_stream`, which yields **only** the text chunks. This is what you'd forward to a UI. Note the `end=""` so chunks print inline as they arrive.

In [4]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

A cloud-based NoSQL database called "Streamline" that automatically optimizes data storage and retrieval for real-time analytics applications, offering seamless scalability across distributed systems while maintaining sub-millisecond query response times.

## 3. Getting the complete message — `get_final_message()`

Stream chunks for the user *and* get the assembled message for storage or further processing. Call `get_final_message()` after the stream finishes.

In [5]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        # In a real app, forward each chunk to the client here.
        pass

    # After streaming, assemble the whole thing:
    final_message = stream.get_final_message()

print("Stop reason: ", final_message.stop_reason)
print("Output tokens:", final_message.usage.output_tokens)
print("\nFull message object:")
final_message

Stop reason:  end_turn
Output tokens: 46

Full message object:


ParsedMessage(id='msg_011Ccw6ciSMC4apYp5x8fHEg', container=None, content=[ParsedTextBlock(citations=None, text='A lightweight, in-memory key-value store called "QuickCache" that simulates database operations for testing purposes without persisting data to disk.', type='text', parsed_output=None)], model='claude-sonnet-5', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=21, output_tokens=46, output_tokens_details=None, server_tool_use=None, service_tier='standard'))